# Querying Project Gutenberg metadata

This notebook queries [Gutendex](https://github.com/garethbjohnson/gutendex), a JSON API backed by Project Gutenberg's catalog metadata. Project Gutenberg itself publishes [official RDF/XML, CSV, and OPDS feeds](https://www.gutenberg.org/ebooks/offline_catalogs.html), but does not provide an official JSON REST API. All dataframe work uses [Polars](https://pola.rs/).

The workflow below covers parameter discovery, filtered queries, pagination, ebook downloads, text cleaning, virtual-page reading, deterministic sampling, retrieval chunks, and Linger-oriented corpus screening. No API key is required.

In [ ]:
# Dependencies are managed from the repository root:
# uv sync --dev
# uv run jupyter lab

In [ ]:
from collections.abc import Iterator, Mapping, Sequence
from datetime import datetime, timezone
from hashlib import sha256
import json
from pathlib import Path
import math
import random
import re
import textwrap
from typing import Any, Optional, Union

import polars as pl
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://gutendex.com/books/"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "gutenberg"
REQUEST_TIMEOUT = (10, 90)  # seconds: connect, read
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "linger-gutenberg-notebook/1.0"})
RETRY_POLICY = Retry(
    total=4,
    connect=4,
    read=4,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}),
)
SESSION.mount("https://", HTTPAdapter(max_retries=RETRY_POLICY))

## Available query parameters

Multiple filters can be combined. Comma-separated values within `ids`, `languages`, and `copyright` mean **any of** those values. Separate parameters are combined as filters.

| Parameter | Meaning | Example |
|---|---|---|
| `search` | Case-insensitive words in author names or titles | `jane austen` |
| `topic` | Case-insensitive phrase in subjects or bookshelves | `children` |
| `languages` | Comma-separated two-letter language codes | `en,fr` |
| `author_year_start` | At least one author was alive on/after this year | `1800` |
| `author_year_end` | At least one author was alive on/before this year | `1899` |
| `copyright` | Copyright status: `true`, `false`, or `null` | `false,null` |
| `ids` | Comma-separated positive Gutenberg IDs | `11,84,1342` |
| `mime_type` | Available format whose MIME type starts with this value | `text/plain` |
| `sort` | `popular`, `ascending`, or `descending` | `popular` |
| `page` | Results page | `2` |

> `author_year_start` and `author_year_end` describe an author's lifespan—not a book's original publication date. Gutenberg generally does not expose original print-publication dates in its machine-readable catalog.

In [ ]:
QUERY_PARAMETERS = {
    "search": "Words in author names or titles",
    "topic": "Phrase in subjects or bookshelves",
    "languages": "Comma-separated two-letter language codes",
    "author_year_start": "Author alive on or after this year",
    "author_year_end": "Author alive on or before this year",
    "copyright": "Comma-separated true, false, or null values",
    "ids": "Comma-separated Gutenberg IDs",
    "mime_type": "Required download MIME-type prefix",
    "sort": "popular, ascending, or descending",
    "page": "Positive results page number",
}
pl.DataFrame({
    "parameter": list(QUERY_PARAMETERS),
    "meaning": list(QUERY_PARAMETERS.values()),
})

## Query one page

Edit `query` and rerun the cells. The helper rejects misspelled parameters instead of silently sending a dubious request.

In [ ]:
def query_books(params: Optional[Mapping[str, Any]] = None) -> dict[str, Any]:
    """Return one page of Gutendex results."""
    params = dict(params or {})
    unknown = set(params) - QUERY_PARAMETERS.keys()
    if unknown:
        raise ValueError(f"Unknown query parameter(s): {sorted(unknown)}")

    response = SESSION.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


query = {
    "search": "Jane Austen",
    "languages": "en",
    "mime_type": "text/plain",
    "sort": "popular",
}
page = query_books(query)
print(f"Matches: {page['count']:,}; records on this page: {len(page['results'])}")
print(f"Next page: {page['next']}")

## Normalize records into a table

Gutendex returns these book fields: `id`, `title`, `authors`, `summaries`, `editors`, `translators`, `subjects`, `bookshelves`, `languages`, `copyright`, `media_type`, `formats`, and `download_count`. Person objects contain `name`, `birth_year`, and `death_year`.

In [ ]:
def join_names(people: Sequence[Mapping[str, Any]]) -> str:
    return "; ".join(person["name"] for person in people)


def books_to_frame(books: Sequence[Mapping[str, Any]]) -> pl.DataFrame:
    rows = []
    for book in books:
        rows.append({
            "id": book["id"],
            "title": book["title"],
            "authors": join_names(book["authors"]),
            "languages": ", ".join(book["languages"]),
            "subjects": "; ".join(book["subjects"]),
            "bookshelves": "; ".join(book["bookshelves"]),
            "copyright": book["copyright"],
            "media_type": book["media_type"],
            "download_count": book["download_count"],
            "available_formats": "; ".join(book["formats"].keys()),
        })
    return pl.DataFrame(rows)


books_df = books_to_frame(page["results"])
books_df

## Follow pagination safely

The public Gutendex instance is suitable for exploration. Keep `max_pages` bounded; for bulk ingestion, download Gutenberg's official catalog and query it locally.

In [ ]:
def iter_books(
    params: Optional[Mapping[str, Any]] = None, *, max_pages: int = 3
) -> Iterator[dict[str, Any]]:
    """Yield books while following at most max_pages result pages."""
    if max_pages < 1:
        raise ValueError("max_pages must be at least 1")

    current = query_books(params)
    for page_number in range(1, max_pages + 1):
        yield from current["results"]
        next_url = current.get("next")
        if not next_url or page_number == max_pages:
            break
        response = SESSION.get(next_url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        current = response.json()


sample_books = list(iter_books({"topic": "memory", "languages": "en"}, max_pages=2))
sample_df = books_to_frame(sample_books)
print(f"Collected {len(sample_df)} records")
sample_df.head()

## Retrieve one book and choose a downloadable format

Format availability varies by book. Prefer UTF-8 plain text for downstream text processing, then fall back to another plain-text representation.

In [ ]:
def get_book(book_id: int) -> dict[str, Any]:
    response = SESSION.get(f"{BASE_URL}{book_id}", timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def choose_text_url(book: Mapping[str, Any]) -> Optional[str]:
    formats = book["formats"]
    preferred = (
        "text/plain; charset=utf-8",
        "text/plain; charset=us-ascii",
        "text/plain",
    )
    for mime_type in preferred:
        if formats.get(mime_type):
            return formats[mime_type]
    return next(
        (url for mime_type, url in formats.items() if mime_type.startswith("text/plain") and url),
        None,
    )


book = get_book(1342)  # Pride and Prejudice
text_url = choose_text_url(book)
{
    "id": book["id"],
    "title": book["title"],
    "authors": join_names(book["authors"]),
    "text_url": text_url,
}

In [ ]:
def download_text(
    book: Mapping[str, Any],
    output_dir: Union[str, Path] = DATA_DIR,
    *,
    overwrite: bool = False,
) -> Path:
    """Download a book's preferred plain-text file and return its local path."""
    url = choose_text_url(book)
    if url is None:
        raise ValueError(f"No plain-text format is available for Gutenberg #{book['id']}")

    destination = Path(output_dir) / f"pg{book['id']}.txt"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        return destination
    response = SESSION.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    destination.write_bytes(response.content)
    metadata_path = destination.with_suffix(".metadata.json")
    metadata_path.write_text(
        json.dumps({
            "gutenberg_id": book["id"],
            "title": book["title"],
            "requested_url": url,
            "resolved_url": response.url,
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
            "sha256": sha256(response.content).hexdigest(),
        }, indent=2),
        encoding="utf-8",
    )
    return destination

local_path = download_text(book)
print(local_path.resolve())

## Separate the ebook body from Gutenberg's wrapper

Keep the downloaded file unchanged as the provenance source. The helper below returns the header, body, and license/footer separately. Marker matching is deliberately conservative: if the markers are absent, it preserves the whole text as the body.

In [ ]:
START_MARKER_RE = re.compile(
    r"\*{3}\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*{3}",
    re.IGNORECASE,
)
END_MARKER_RE = re.compile(
    r"\*{3}\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*{3}",
    re.IGNORECASE,
)


def read_downloaded_text(path: Union[str, Path]) -> str:
    return Path(path).read_text(encoding="utf-8-sig", errors="replace").replace("\r\n", "\n")


def split_gutenberg_text(raw_text: str) -> tuple[str, str, str]:
    start = START_MARKER_RE.search(raw_text)
    body_start = start.end() if start else 0
    end = END_MARKER_RE.search(raw_text, body_start)
    body_end = end.start() if end else len(raw_text)
    return (
        raw_text[:body_start].strip(),
        raw_text[body_start:body_end].strip(),
        raw_text[body_end:].strip(),
    )


raw_text = read_downloaded_text(local_path)
header, body_text, footer = split_gutenberg_text(raw_text)
print(f"Raw characters: {len(raw_text):,}")
print(f"Body characters: {len(body_text):,}")

## Read virtual pages and deterministic samples

Plain text does **not** retain dependable page numbers from a print edition. These helpers therefore use clearly labelled virtual pages and word offsets. They are useful for inspection, but Linger citations should ultimately point to a versioned local text plus chapter/section and stable offsets—not pretend these are print pages.

In [ ]:
def virtual_page(
    text: str, page_number: int, *, words_per_page: int = 300, width: int = 100
) -> str:
    if page_number < 1:
        raise ValueError("page_number starts at 1")
    words = text.split()
    start = (page_number - 1) * words_per_page
    if start >= len(words):
        raise IndexError(f"Page {page_number} is beyond the {math.ceil(len(words) / words_per_page)} virtual pages")
    excerpt = " ".join(words[start : start + words_per_page])
    return textwrap.fill(excerpt, width=width)


def read_virtual_pages(
    text: str, *, start_page: int = 1, page_count: int = 2, words_per_page: int = 300
) -> None:
    for page_number in range(start_page, start_page + page_count):
        print(f"\n=== Virtual page {page_number} ===\n")
        print(virtual_page(text, page_number, words_per_page=words_per_page))


def sample_passages(
    text: str, *, samples: int = 3, passage_words: int = 220, seed: int = 42
) -> pl.DataFrame:
    words = text.split()
    if len(words) < passage_words:
        raise ValueError("The text is shorter than one requested passage")
    margin = min(
        len(words) // 20, 2_000, max((len(words) - passage_words) // 2, 0)
    )  # avoid sampling only front/back matter
    low = margin
    high = max(low, len(words) - margin - passage_words)
    rng = random.Random(seed)
    starts = sorted(rng.randint(low, high) for _ in range(samples))
    return pl.DataFrame([
        {
            "sample": index + 1,
            "word_start": start,
            "word_end": start + passage_words,
            "text": " ".join(words[start : start + passage_words]),
        }
        for index, start in enumerate(starts)
    ])


read_virtual_pages(body_text, start_page=2, page_count=2)

samples_df = sample_passages(body_text)
for row in samples_df.iter_rows(named=True):
    print(f"\n--- Sample {row['sample']} (words {row['word_start']:,}–{row['word_end']:,}) ---\n")
    print(textwrap.fill(row["text"], width=100))

## Build retrieval-sized chunks

Fixed word windows are a baseline, not the final chunker. For Linger, prefer chapter/section-aware chunks where headings are reliable, retain overlap for boundary recall, and store the source checksum so offsets remain tied to the exact downloaded version.

In [ ]:
def chunk_book(
    book_id: int, text: str, *, chunk_words: int = 500, overlap_words: int = 100
) -> pl.DataFrame:
    if chunk_words < 1 or not 0 <= overlap_words < chunk_words:
        raise ValueError("Require chunk_words > 0 and 0 <= overlap_words < chunk_words")
    words = text.split()
    step = chunk_words - overlap_words
    source_hash = sha256(text.encode("utf-8")).hexdigest()
    rows = []
    for chunk_index, start in enumerate(range(0, len(words), step)):
        end = min(start + chunk_words, len(words))
        rows.append({
            "chunk_id": f"pg{book_id}:{source_hash[:12]}:w{start}-{end}",
            "book_id": book_id,
            "source_sha256": source_hash,
            "word_start": start,
            "word_end": end,
            "text": " ".join(words[start:end]),
        })
        if end == len(words):
            break
    return pl.DataFrame(rows)


chunks_df = chunk_book(book["id"], body_text)
chunks_df.select("chunk_id", "word_start", "word_end").head()

## Profile corpus suitability

Gutendex cannot filter by word count, chapter structure, or OCR quality. Those require downloading candidate texts and profiling them locally. Gutenberg texts are normally proofread—even when OCR was part of production—so the checks below flag **possible residual damage for review** rather than claiming to measure OCR accuracy.

Useful Linger characteristics include:

- moderate body length: enough retrieval cases without one title dominating the corpus;
- chapter/book/part headings: useful for spoiler boundaries and human-readable citations;
- low replacement-character, mojibake, mixed letter/digit, and line-end hyphen signals;
- prose/verse and paragraph-shape indicators: not quality judgments, but chunking warnings;
- a stable Gutenberg ID, source URL, and SHA-256 checksum;
- subject, genre, author, era, and style diversity across the final 3–5 books; and
- manual review of sample passages, edition/translation, cultural framing, and duplicate editions.

The default 40k–160k word range is editable. The comparison set deliberately includes a short work, a prose translation of the *Odyssey* as an edition/section-structure stress case, and *Moby-Dick* as a long-book stress case.

In [ ]:
PROFILE_SETTINGS = {
    "min_words": 40_000,
    "max_words": 160_000,
    "words_per_estimated_page": 300,
    "min_structure_headings": 5,
    "max_replacement_chars_per_100k": 1.0,
    "max_mojibake_markers_per_100k": 1.0,
    "max_mixed_alnum_tokens_per_10k": 5.0,
    "max_line_end_hyphens_per_10k": 12.0,
}

CHAPTER_RE = re.compile(
    r"(?im)^\s*(?:chapter|book|part|canto)\s+(?:[ivxlcdm]+|\d+|[a-z]+)\b[^\n]*$"
)
MOJIBAKE_MARKERS = ("Ã", "Â", "â€™", "â€œ", "â€", "ï»¿")


def per_n(count: int, denominator: int, scale: int) -> float:
    return round(count * scale / max(denominator, 1), 3)


def profile_book_text(
    book: Mapping[str, Any], raw_text: str, body: str, source_path: Union[str, Path]
) -> dict[str, Any]:
    words = re.findall(r"\b[\w’'-]+\b", body, flags=re.UNICODE)
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    paragraphs = [p for p in re.split(r"\n\s*\n", body) if p.strip()]
    mixed_alnum = sum(
        bool(re.search(r"[A-Za-z]\d|\d[A-Za-z]", token)) for token in words
    )
    line_end_hyphens = len(re.findall(r"[A-Za-z]{2,}-\n[a-z]{2,}", body))
    replacements = body.count("�")
    mojibake = sum(body.count(marker) for marker in MOJIBAKE_MARKERS)
    headings = CHAPTER_RE.findall(body)
    word_count = len(words)
    retrieval_token_count = len(body.split())
    paragraph_word_counts = [len(p.split()) for p in paragraphs]
    source_url = choose_text_url(book)
    metadata_path = Path(source_path).with_suffix(".metadata.json")
    download_metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    start_marker = START_MARKER_RE.search(raw_text)
    lower_header = raw_text[: start_marker.end() if start_marker else 5_000].lower()
    sentences = [s for s in re.split(r"[.!?]+(?:[\"'’”)]*)\s+", body) if s.strip()]

    return {
        "id": book["id"],
        "title": book["title"],
        "authors": join_names(book["authors"]),
        "languages": ",".join(book["languages"]),
        "subjects": "; ".join(book["subjects"]),
        "word_count": word_count,
        "estimated_pages": math.ceil(word_count / PROFILE_SETTINGS["words_per_estimated_page"]),
        "retrieval_token_count": retrieval_token_count,
        "retrieval_chunks_500_100": max(1, math.ceil(max(retrieval_token_count - 500, 0) / 400) + 1),
        "mean_sentence_words": round(word_count / max(len(sentences), 1), 1),
        "structure_headings": len(headings),
        "paragraphs": len(paragraphs),
        "median_paragraph_words": float(
            sorted(paragraph_word_counts)[len(paragraph_word_counts) // 2]
        ) if paragraph_word_counts else 0.0,
        "short_line_ratio": round(sum(len(line.split()) <= 4 for line in lines) / max(len(lines), 1), 3),
        "replacement_chars_per_100k": per_n(replacements, len(body), 100_000),
        "mojibake_markers_per_100k": per_n(mojibake, len(body), 100_000),
        "mixed_alnum_tokens_per_10k": per_n(mixed_alnum, word_count, 10_000),
        "line_end_hyphens_per_10k": per_n(line_end_hyphens, word_count, 10_000),
        "ocr_mentioned_in_wrapper": "ocr" in lower_header or "optical character" in lower_header,
        "distributed_proofreaders_mentioned": "distributed proofreaders" in lower_header,
        "download_count_30d": book["download_count"],
        "source_url": source_url,
        "source_path": str(Path(source_path)),
        "source_sha256": sha256(Path(source_path).read_bytes()).hexdigest(),
        "downloaded_at_utc": download_metadata.get("downloaded_at_utc"),
        "profiled_at_utc": datetime.now(timezone.utc).isoformat(),
    }


In [ ]:
# Edit this comparison set or replace it with IDs found through a Gutendex query.
CANDIDATE_IDS = [11, 84, 1342, 1661, 1727, 2701]
candidate_records = query_books({"ids": ",".join(map(str, CANDIDATE_IDS))})["results"]
candidate_by_id = {record["id"]: record for record in candidate_records}

profiles = []
candidate_texts = {}
for book_id in CANDIDATE_IDS:
    candidate = candidate_by_id[book_id]
    path = download_text(candidate)
    raw = read_downloaded_text(path)
    _, candidate_body, _ = split_gutenberg_text(raw)
    candidate_texts[book_id] = candidate_body
    profiles.append(profile_book_text(candidate, raw, candidate_body, path))

profiles_df = (
    pl.DataFrame(profiles)
    .with_columns(
        pl.col("word_count").is_between(
            PROFILE_SETTINGS["min_words"], PROFILE_SETTINGS["max_words"], closed="both"
        ).alias("length_ok"),
        (pl.col("structure_headings") >= PROFILE_SETTINGS["min_structure_headings"]).alias("structure_ok"),
        (
            (pl.col("replacement_chars_per_100k") <= PROFILE_SETTINGS["max_replacement_chars_per_100k"])
            & (pl.col("mojibake_markers_per_100k") <= PROFILE_SETTINGS["max_mojibake_markers_per_100k"])
            & (pl.col("mixed_alnum_tokens_per_10k") <= PROFILE_SETTINGS["max_mixed_alnum_tokens_per_10k"])
            & (pl.col("line_end_hyphens_per_10k") <= PROFILE_SETTINGS["max_line_end_hyphens_per_10k"])
        ).alias("text_quality_checks_ok"),
    )
    .with_columns(
        (pl.col("length_ok") & pl.col("structure_ok") & pl.col("text_quality_checks_ok")).alias("automatic_shortlist"),
        (pl.col("retrieval_chunks_500_100") / pl.col("retrieval_chunks_500_100").sum()).round(3).alias("candidate_chunk_share"),
    )
    .sort(["automatic_shortlist", "word_count"], descending=[True, False])
)

profiles_df.select(
    "id", "title", "word_count", "estimated_pages", "retrieval_chunks_500_100",
    "candidate_chunk_share", "mean_sentence_words", "structure_headings",
    "short_line_ratio", "ocr_mentioned_in_wrapper", "length_ok",
    "structure_ok", "text_quality_checks_ok", "automatic_shortlist",
)

## Review flagged metrics and samples before selecting

`automatic_shortlist` is intentionally conservative and should never be the final decision. A verse work may have many short lines yet be perfectly clean; an OCR-derived book may be excellently proofread; chapter detection may miss unconventional headings. Inspect the component metrics and several passages from every shortlisted book.

In [ ]:
quality_review_df = profiles_df.select(
    "id", "title", "replacement_chars_per_100k", "mojibake_markers_per_100k",
    "mixed_alnum_tokens_per_10k", "line_end_hyphens_per_10k",
    "short_line_ratio", "source_sha256",
)
quality_review_df

In [ ]:
BOOK_TO_REVIEW = 1342
review_samples = sample_passages(candidate_texts[BOOK_TO_REVIEW], samples=4, passage_words=180)
for row in review_samples.iter_rows(named=True):
    print(f"\n--- PG #{BOOK_TO_REVIEW}, sample {row['sample']} "
          f"(words {row['word_start']:,}–{row['word_end']:,}) ---\n")
    print(textwrap.fill(row["text"], width=100))

## Final Linger selection checklist

Before choosing 3–5 books, check what automation cannot decide:

1. **Retrieval usefulness:** enough semantically similar scenes, ideas, and recurring motifs to create hard positive/negative cases.
2. **Spoiler control:** chapter or section boundaries that can map cleanly to a user's reading position.
3. **Corpus balance:** varied authors, eras, genres, prose styles, themes, and difficulty; avoid one very long book dominating chunks.
4. **Edition identity:** confirm the translation and edition; Gutenberg may carry multiple versions of works such as the *Odyssey*.
5. **Text shape:** tables, footnotes, poetry, drama, illustrations, and dialect may need specialized chunking rather than rejection.
6. **Text quality:** manually inspect beginning, middle, end, and deterministic random samples; heuristics only surface risk.
7. **Provenance:** retain the untouched download, Gutenberg ID, exact source URL, checksum, and retrieval date.
8. **Rights and framing:** verify copyright for the deployment jurisdiction and disclose dated or harmful cultural perspectives.

After review, uncomment the export below to preserve the candidate audit table.

In [ ]:
# audit_path = DATA_DIR / "candidate_profiles.parquet"
# audit_path.parent.mkdir(parents=True, exist_ok=True)
# profiles_df.write_parquet(audit_path)
# print(audit_path.resolve())

## Useful query recipes

```python
# Several known books
query_books({"ids": "11,84,1342"})

# Public-domain English detective fiction with EPUB downloads
query_books({
    "topic": "detective fiction",
    "languages": "en",
    "copyright": "false",
    "mime_type": "application/epub+zip",
})

# Oldest Gutenberg IDs first (not oldest original books)
query_books({"languages": "en", "sort": "ascending"})
```

For a small curated Linger corpus, record the Gutenberg ID, title, authors, language, chosen source URL, retrieval timestamp, and a checksum alongside each downloaded text. That gives later quotations stable provenance even if upstream metadata changes.